In [ ]:
# ─── H1: Imóveis com acesso à orla (waterfront) têm preço médio significativamente maior ───

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('H1 — Acesso à Orla (waterfront) vs Preço', fontsize=14, fontweight='bold')

# Boxplot
sns.boxplot(data=df, x='waterfront', y='price', palette=['#5b9bd5', '#1f5fa6'], ax=axes[0])
axes[0].set_xticklabels(['Sem orla', 'Com orla'])
axes[0].set_title('Distribuição de Preços')
axes[0].set_ylabel('Preço (USD)')
axes[0].set_xlabel('')

# Preço médio em barras
media_wf = df.groupby('waterfront')['price'].mean()
barras = axes[1].bar(['Sem orla', 'Com orla'], media_wf.values,
                      color=['#5b9bd5', '#1f5fa6'], edgecolor='white', width=0.5)
axes[1].set_title('Preço Médio por Acesso à Orla')
axes[1].set_ylabel('Preço Médio (USD)')
for b, v in zip(barras, media_wf.values):
    axes[1].text(b.get_x() + b.get_width()/2, v + 15000, f'${v/1e6:.2f}M',
                 ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('imagens/h1_waterfront.png', dpi=150, bbox_inches='tight')
plt.show()

sem, com_ = media_wf[0], media_wf[1]
print(f"Preço médio SEM orla: ${sem:,.0f}")
print(f"Preço médio COM orla: ${com_:,.0f}")
print(f"► Diferença: {(com_/sem - 1)*100:.1f}% mais caro → H1 CONFIRMADA")

In [ ]:
# ─── H2: Quanto maior a área habitável (sqft_living), maior o preço ───

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('H2 — Área Habitável (sqft_living) vs Preço', fontsize=14, fontweight='bold')

# Dispersão com linha de tendência
axes[0].scatter(df['sqft_living'], df['price'], alpha=0.2, s=4, color='steelblue')
coef = np.polyfit(df['sqft_living'], df['price'], 1)
x_r = np.linspace(df['sqft_living'].min(), df['sqft_living'].max(), 200)
axes[0].plot(x_r, np.polyval(coef, x_r), color='red', linewidth=2, label='Tendência linear')
axes[0].set_title('Dispersão: Área vs Preço')
axes[0].set_xlabel('Área Habitável (sqft)')
axes[0].set_ylabel('Preço (USD)')
axes[0].legend()

# Preço médio por faixa de área
df['faixa_area'] = pd.cut(df['sqft_living'],
                           bins=[0, 1000, 1500, 2000, 2500, 3000, 4000, 14000],
                           labels=['<1k', '1-1.5k', '1.5-2k', '2-2.5k', '2.5-3k', '3-4k', '>4k'])
media_faixa = df.groupby('faixa_area', observed=True)['price'].mean()
cores = sns.color_palette("Blues_r", len(media_faixa))
barras = axes[1].bar(media_faixa.index, media_faixa.values, color=cores, edgecolor='white')
axes[1].set_title('Preço Médio por Faixa de Área')
axes[1].set_xlabel('Faixa de Área (sqft)')
axes[1].set_ylabel('Preço Médio (USD)')
axes[1].tick_params(axis='x', rotation=20)
for b, v in zip(barras, media_faixa.values):
    axes[1].text(b.get_x() + b.get_width()/2, v + 8000, f'${v/1e3:.0f}k', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('imagens/h2_sqft_living.png', dpi=150, bbox_inches='tight')
plt.show()

corr = df['sqft_living'].corr(df['price'])
print(f"Correlação de Pearson sqft_living vs price: r = {corr:.4f}")
print(f"► Correlação positiva {'forte' if corr > 0.6 else 'moderada'} → H2 CONFIRMADA")

In [ ]:
# ─── H3: O grau de construção (grade) tem a maior correlação com o preço ───

# Calculando correlações de todas as variáveis numéricas com o preço
correlacoes = df[['sqft_living', 'grade', 'bathrooms', 'bedrooms',
                   'view', 'condition', 'floors', 'yr_built']].corrwith(df['price']).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('H3 — Grau de Construção (grade) vs Preço', fontsize=14, fontweight='bold')

# Ranking de correlações
cores_rank = ['#d73027' if c == 'grade' else '#74add1' for c in correlacoes.index]
axes[0].barh(correlacoes.index[::-1], correlacoes.values[::-1], color=cores_rank[::-1])
axes[0].set_title('Correlação das Variáveis com Preço (|r|)')
axes[0].set_xlabel('|Correlação de Pearson|')
for i, (var, val) in enumerate(zip(correlacoes.index[::-1], correlacoes.values[::-1])):
    axes[0].text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=9)

# Boxplot grade vs preço (grades mais comuns)
df_grade = df[df['grade'].between(4, 12)]
sns.boxplot(data=df_grade, x='grade', y='price', palette='YlOrRd', ax=axes[1])
axes[1].set_title('Distribuição de Preço por Grade (4–12)')
axes[1].set_xlabel('Grade de Construção')
axes[1].set_ylabel('Preço (USD)')

plt.tight_layout()
plt.savefig('imagens/h3_grade.png', dpi=150, bbox_inches='tight')
plt.show()

print("Correlações com o preço (ordenadas):")
for var, val in correlacoes.items():
    marca = "  ◄ MAIOR" if var == correlacoes.index[0] else ""
    print(f"  {var:15s}: r = {val:.4f}{marca}")
print(f"\n► grade {'É' if correlacoes.index[0]=='grade' else 'NÃO É'} a variável de maior correlação → H3 {'CONFIRMADA' if correlacoes.index[0]=='grade' else 'PARCIALMENTE CONFIRMADA'}")

In [ ]:
# ─── H4: Imóveis renovados têm preço médio maior do que os não renovados ───

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('H4 — Renovação vs Preço', fontsize=14, fontweight='bold')

# Boxplot
sns.boxplot(data=df, x='renovado', y='price', palette=['#66c2a5', '#fc8d62'], ax=axes[0])
axes[0].set_xticklabels(['Não renovado', 'Renovado'])
axes[0].set_title('Distribuição de Preços por Renovação')
axes[0].set_ylabel('Preço (USD)')
axes[0].set_xlabel('')

# Histogramas sobrepostos
for status, label, cor in [(0, 'Não renovado', '#66c2a5'), (1, 'Renovado', '#fc8d62')]:
    dados = df[df['renovado'] == status]['price']
    axes[1].hist(dados, bins=60, alpha=0.55,
                 label=f'{label} (n={len(dados):,})', color=cor, density=True)
m_nao = df[df['renovado'] == 0]['price'].mean()
m_sim = df[df['renovado'] == 1]['price'].mean()
axes[1].axvline(m_nao, color='#3aa882', linestyle='--', linewidth=2, label=f'Média não renovado: ${m_nao/1e3:.0f}k')
axes[1].axvline(m_sim, color='#e05a20', linestyle='--', linewidth=2, label=f'Média renovado: ${m_sim/1e3:.0f}k')
axes[1].set_title('Distribuição de Preços (densidade)')
axes[1].set_xlabel('Preço (USD)')
axes[1].set_ylabel('Densidade')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('imagens/h4_renovacao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Preço médio NÃO renovado: ${m_nao:,.0f}")
print(f"Preço médio RENOVADO:     ${m_sim:,.0f}")
print(f"► Renovados custam {(m_sim/m_nao - 1)*100:.1f}% mais em média → H4 CONFIRMADA")

In [ ]:
# ─── H5: Banheiros têm correlação mais forte com o preço do que quartos ───

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('H5 — Banheiros vs Quartos: qual tem maior correlação com o preço?', fontsize=13, fontweight='bold')

# Banheiros vs preço médio
df_bath = df[df['bathrooms'] <= 6]
media_bath = df_bath.groupby('bathrooms')['price'].mean()
axes[0].scatter(media_bath.index, media_bath.values, s=100, color='coral', zorder=5)
coef_b = np.polyfit(df_bath['bathrooms'], df_bath['price'], 1)
x_b = np.linspace(df_bath['bathrooms'].min(), df_bath['bathrooms'].max(), 100)
axes[0].plot(x_b, np.polyval(coef_b, x_b), 'r--', linewidth=2)
r_bath = df['bathrooms'].corr(df['price'])
axes[0].set_title(f'Banheiros vs Preço Médio  (r = {r_bath:.3f})')
axes[0].set_xlabel('Número de Banheiros')
axes[0].set_ylabel('Preço Médio (USD)')

# Quartos vs preço médio
df_bed = df[df['bedrooms'].between(1, 8)]
media_bed = df_bed.groupby('bedrooms')['price'].mean()
axes[1].scatter(media_bed.index, media_bed.values, s=100, color='steelblue', zorder=5)
coef_r = np.polyfit(df_bed['bedrooms'], df_bed['price'], 1)
x_r = np.linspace(1, 8, 100)
axes[1].plot(x_r, np.polyval(coef_r, x_r), 'b--', linewidth=2)
r_bed = df['bedrooms'].corr(df['price'])
axes[1].set_title(f'Quartos vs Preço Médio  (r = {r_bed:.3f})')
axes[1].set_xlabel('Número de Quartos')
axes[1].set_ylabel('Preço Médio (USD)')

plt.tight_layout()
plt.savefig('imagens/h5_bath_bed.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Correlação BANHEIROS vs preço: r = {r_bath:.4f}")
print(f"Correlação QUARTOS   vs preço: r = {r_bed:.4f}")
vencedor = 'Banheiros' if r_bath > r_bed else 'Quartos'
print(f"► {vencedor} têm correlação mais forte → H5 {'CONFIRMADA' if r_bath > r_bed else 'REJEITADA'}")

In [ ]:
# ─── H6 a H10: Hipóteses Complementares ───

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('H6–H10: Hipóteses Complementares', fontsize=14, fontweight='bold')

# H6 — Porão
sns.boxplot(data=df, x='tem_porao', y='price', palette='Pastel1', ax=axes[0, 0])
axes[0, 0].set_xticklabels(['Sem porão', 'Com porão'])
axes[0, 0].set_title('H6 — Porão vs Preço')
axes[0, 0].set_ylabel('Preço (USD)')
axes[0, 0].set_xlabel('')

# H7 — Andares
media_floors = df.groupby('floors')['price'].mean()
axes[0, 1].bar(media_floors.index.astype(str), media_floors.values,
               color=sns.color_palette("Blues_r", len(media_floors)))
axes[0, 1].set_title('H7 — Andares vs Preço Médio')
axes[0, 1].set_xlabel('Andares')
axes[0, 1].set_ylabel('Preço Médio (USD)')

# H8 — Vista
media_view = df.groupby('view')['price'].mean()
axes[0, 2].bar(media_view.index.astype(str), media_view.values,
               color=sns.color_palette("YlGnBu", len(media_view)))
axes[0, 2].set_title('H8 — Vista vs Preço Médio')
axes[0, 2].set_xlabel('Índice de Vista (0–4)')
axes[0, 2].set_ylabel('Preço Médio (USD)')

# H9 — Condição
media_cond = df.groupby('condition')['price'].mean()
axes[1, 0].bar(media_cond.index.astype(str), media_cond.values,
               color=sns.color_palette("Greens", len(media_cond)))
axes[1, 0].set_title('H9 — Condição vs Preço Médio')
axes[1, 0].set_xlabel('Condição (1–5)')
axes[1, 0].set_ylabel('Preço Médio (USD)')

# H10 — Década de construção
media_dec = df.groupby('decada')['price'].mean()
axes[1, 1].plot(media_dec.index, media_dec.values, marker='o', color='steelblue', linewidth=2)
axes[1, 1].fill_between(media_dec.index, media_dec.values, alpha=0.2, color='steelblue')
axes[1, 1].set_title('H10 — Década de Construção vs Preço')
axes[1, 1].set_xlabel('Década')
axes[1, 1].set_ylabel('Preço Médio (USD)')
axes[1, 1].tick_params(axis='x', rotation=30)

# Heatmap de correlações (resumo geral)
corr_vars = ['price', 'sqft_living', 'grade', 'bathrooms', 'bedrooms', 'view', 'condition', 'floors', 'yr_built']
corr_mat = df[corr_vars].corr()
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[1, 2], linewidths=0.5, annot_kws={'size': 7})
axes[1, 2].set_title('Mapa de Correlações Geral')

plt.tight_layout()
plt.savefig('imagens/h6_h10_complementares.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumo numérico
r6 = df[df['tem_porao'] == 1]['price'].mean() / df[df['tem_porao'] == 0]['price'].mean()
print(f"H6  — Porão: imóveis com porão custam {(r6 - 1)*100:.1f}% mais")
print(f"H7  — Andares: r = {df['floors'].corr(df['price']):.4f}")
print(f"H8  — Vista:   r = {df['view'].corr(df['price']):.4f}")
print(f"H9  — Condição: r = {df['condition'].corr(df['price']):.4f} (correlação fraca)")
print(f"H10 — Ano construção: r = {df['yr_built'].corr(df['price']):.4f} (relação não-linear)")